# Data audit

Inspect source schemas, data quality and possible connections between the datasets.

In [1]:
from pathlib import Path
import sys
import json

import pandas as pd
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/config.py").exists())
sys.path.insert(0, str(ROOT))
from src import config
from src.data import write_json

pd.set_option("display.max_columns", 20)

from src.data import download_metropt, load_metropt, load_maintenance, validate_equipment_id, sha256
from src.text_extraction import clean_text

for directory in ["data/raw", "data/processed", "results"]:
    (ROOT / directory).mkdir(parents=True, exist_ok=True)
raw = ROOT / "data/raw"
metro_path = download_metropt(raw)
maintenance_path = raw / "maintenance_sample.parquet"
snapshot = json.loads((raw / "source_snapshot.json").read_text(encoding="utf-8"))
assert sha256(metro_path) == snapshot["metropt3_sha256"], "MetroPT snapshot changed."
assert sha256(maintenance_path) == snapshot["maintenance_sha256"], "Maintenance snapshot changed."
sensor = load_metropt(metro_path)
maintenance = load_maintenance(maintenance_path)
display(sensor.head(3))
display(maintenance.head(3))

,timestamp,TP2,TP3,H1,DV_pressure,Reservoirs,Oil_temperature,Motor_current,COMP,DV_eletric,Towers,MPG,LPS,Pressure_switch,Oil_level,Caudal_impulses
0,2020-02-01 00:00:00,-0.012,9.358,9.340,-0.024,9.358,53.600,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
1,2020-02-01 00:00:10,-0.014,9.348,9.332,-0.022,9.348,53.675,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
2,2020-02-01 00:00:19,-0.012,9.338,9.322,-0.022,9.338,53.600,0.0425,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0


,row_id,Location (building number),Region,WorkOrder,Equipment_ID,OrderType,WorkOrderDescription,OperationDescription,MaintenanceWorkCenter,OperationWorkCenter,...,CreatedDate,ScheduledStartDate,ScheduledFinishDate,ActualStartDate,ActualFinishDate,TechnicalCompletionDate,CloseData,ActualWork,PlannedWork,OrderStatus
0,0,B04,SOUTH,20346569,M-067,N/A,Motor M-02 earth fault trip. Single phase to e...,Isolate motor. Identify faulted phase using IR...,ELEC_TEAM_A,ELEC_TEAM_A,...,2019-03-07,2019-03-09,2019-03-11,2019-03-08,2019-03-08,07/03/2019,2019-03-08,11.5,31.2,
1,1,B01,EAST,20381028,P-573,PM,Centrifugal Pump P-501 quarterly lubrication s...,Grease drive end and non-drive end bearings. M...,MECH_TEAM_B,MECH_TEAM_A,...,2019-03-15,2019-0-25,2019-04-26,2019-04-25,2019-04-25,2019-04-24,2019-04-25,16.2,11.4,TECO
2,2,B01,CENTRAL,20232132,SC-045,PM,Screw Conveyor SC-02 annual trough lienr inspe...,Inspect trough liner plates for wear. Replace ...,MECH_TEAM_C,MECH_TEAM_D,...,2019-02-14,2019-03-19,2019-03-20,2019-03-18,2019-03-18,2019-03-18,2019-03-19,18.7,20.0,CRTD


## Structure and quality

In [2]:
intervals = sensor["timestamp"].diff().dt.total_seconds()
sensor_audit = pd.DataFrame([{
    "dataset": "metropt3", "rows": len(sensor), "columns": sensor.shape[1],
    "missing_cells": int(sensor.isna().sum().sum()),
    "duplicate_rows": int(sensor.duplicated().sum()),
    "duplicate_timestamps": int(sensor["timestamp"].duplicated().sum()),
    "timestamp_min": str(sensor["timestamp"].min()), "timestamp_max": str(sensor["timestamp"].max()),
    "median_interval_seconds": float(intervals.median()),
    "memory_mb": float(sensor.memory_usage(deep=True).sum() / 1024**2),
    "analog_channels": len(config.ANALOG), "digital_channels": len(config.DIGITAL)
}])
channel_audit = pd.DataFrame([
    {"channel": column, "kind": "analog" if column in config.ANALOG else "digital",
     "missing": int(sensor[column].isna().sum()), "unique_values": sensor[column].nunique(),
     "minimum": float(sensor[column].min()), "maximum": float(sensor[column].max())}
    for column in (*config.ANALOG, *config.DIGITAL)
])
assert sensor["timestamp"].is_monotonic_increasing
assert sensor[list(config.DIGITAL)].isin([0, 1]).all().all(), "Unexpected digital regime."
display(sensor_audit)
display(channel_audit)

,dataset,rows,columns,missing_cells,duplicate_rows,duplicate_timestamps,timestamp_min,timestamp_max,median_interval_seconds,memory_mb,analog_channels,digital_channels
0,metropt3,1516948,16,0,0,0,2020-02-01 00:00:00,2020-09-01 03:59:50,10.0,185.174442,7,8


,channel,kind,missing,unique_values,minimum,maximum
0,TP2,analog,0,5257,-0.032,10.676
1,TP3,analog,0,3683,0.730,10.302
2,H1,analog,0,2665,-0.036,10.288
3,DV_pressure,analog,0,2257,-0.032,9.844
4,Reservoirs,analog,0,3682,0.712,10.300
5,Oil_temperature,analog,0,2462,15.400,89.050
6,Motor_current,analog,0,1809,0.020,9.295
7,COMP,digital,0,2,0.000,1.000
8,DV_eletric,digital,0,2,0.000,1.000
9,Towers,digital,0,2,0.000,1.000


In [3]:
text = maintenance["WorkOrderDescription"].map(clean_text).str.cat(
    maintenance["OperationDescription"].map(clean_text), sep=" ").str.strip()
valid_ids = maintenance["Equipment_ID"].map(validate_equipment_id)
maintenance_audit = pd.DataFrame([{
    "dataset": "maintenance", "rows": len(maintenance), "columns": maintenance.shape[1],
    "missing_cells": int(maintenance.isna().sum().sum()),
    "duplicate_rows": int(maintenance.drop(columns="row_id").duplicated().sum()),
    "empty_text": int(text.eq("").sum()), "median_text_length": float(text.str.len().median()),
    "valid_id_records": int(valid_ids.ne("").sum()), "valid_assets": valid_ids[valid_ids.ne("")].nunique(),
    "unique_work_orders": maintenance["WorkOrder"].map(clean_text).nunique(),
    "memory_mb": float(maintenance.memory_usage(deep=True).sum() / 1024**2)
}])
missingness = []
for dataset, frame in [("metropt3", sensor), ("maintenance", maintenance)]:
    for column in frame:
        missing = frame[column].isna()
        if dataset == "maintenance" and column != "row_id":
            missing |= frame[column].map(clean_text).eq("")
        missingness.append({"dataset": dataset, "column": column, "missing_or_placeholder": int(missing.sum()),
                            "fraction": float(missing.mean())})
dates = []
for column in [c for c in maintenance if "Date" in c or c == "CloseData"]:
    values = maintenance[column].map(clean_text)
    iso = values.str.fullmatch(r"\d{4}-\d{2}-\d{2}")
    slash = values.str.fullmatch(r"\d{2}/\d{2}/\d{4}")
    parsed = pd.Series(pd.NaT, index=values.index, dtype="datetime64[us]")
    parsed.loc[iso] = pd.to_datetime(values[iso], format="%Y-%m-%d", errors="coerce")
    parsed.loc[slash] = pd.to_datetime(values[slash], format="%d/%m/%Y", errors="coerce")
    dates.append({"field": column, "nonempty": int(values.ne("").sum()),
                  "unparseable": int((values.ne("") & parsed.isna()).sum()),
                  "outside_1900_2100": int((parsed.lt("1900-01-01") | parsed.gt("2100-12-31")).sum()),
                  "slash_dates_day_first": int(slash.sum()),
                  "minimum": str(parsed.min()), "maximum": str(parsed.max())})
audit = pd.concat([sensor_audit, maintenance_audit], ignore_index=True)
display(maintenance_audit)
display(pd.DataFrame(dates))

,dataset,rows,columns,missing_cells,duplicate_rows,empty_text,median_text_length,valid_id_records,valid_assets,unique_work_orders,memory_mb
0,maintenance,50000,21,2399,1150,20,250.0,48678,9400,47600,26.143641


,field,nonempty,unparseable,outside_1900_2100,slash_dates_day_first,minimum,maximum
0,CreatedDate,49010,1085,44,195,0219-01-08 00:00:00,2109-05-12 00:00:00
1,ScheduledStartDate,48975,1114,48,215,0219-01-04 00:00:00,2109-06-03 00:00:00
2,ScheduledFinishDate,49017,1065,65,199,0219-01-10 00:00:00,2109-06-13 00:00:00
3,ActualStartDate,49014,1100,67,210,0219-01-02 00:00:00,2109-07-04 00:00:00
4,ActualFinishDate,49022,1122,60,216,0219-01-04 00:00:00,2109-06-23 00:00:00
5,TechnicalCompletionDate,48973,1034,53,190,0219-01-12 00:00:00,2109-07-12 00:00:00
6,CloseData,48989,634,51,0,0219-01-15 00:00:00,2109-05-25 00:00:00


## Source alignment

The sources have separate asset namespaces and no shared physical asset identifiers. Temporal readings and maintenance descriptions can contribute separate branches to the same industrial ontology.

In [4]:
alignment = pd.DataFrame([
    {"concept": "Asset", "metropt3": "One real compressor APU", "maintenance": "Many synthetic Equipment_ID values", "integration": "shared industrial classes"},
    {"concept": "Time", "metropt3": "2020 sensor timestamps", "maintenance": "synthetic lifecycle dates", "integration": "independent timelines"},
    {"concept": "Observation", "metropt3": "15 sensor channels", "maintenance": "descriptions and structured fields", "integration": "separate source branches"},
    {"concept": "Identity", "metropt3": "MetroPT APU", "maintenance": "synthetic assets", "integration": "direct record-level joins invalid"}
])
valid = "independent source branches integrated through a shared industrial ontology."
invalid = "direct record-level fusion between MetroPT-3 and maintenance records."
display(alignment)
print("Valid:", valid)
print("Invalid:", invalid)

,concept,metropt3,maintenance,integration
0,Asset,One real compressor APU,Many synthetic Equipment_ID values,shared industrial classes
1,Time,2020 sensor timestamps,synthetic lifecycle dates,independent timelines
2,Observation,15 sensor channels,descriptions and structured fields,separate source branches
3,Identity,MetroPT APU,synthetic assets,direct record-level joins invalid


Valid: independent source branches integrated through a shared industrial ontology.
Invalid: direct record-level fusion between MetroPT-3 and maintenance records.


In [5]:
sensor.to_parquet(ROOT / "data/processed/metropt3.parquet", index=False)
maintenance.to_parquet(ROOT / "data/processed/maintenance.parquet", index=False)
audit.to_csv(ROOT / "results/data_audit.csv", index=False)
channel_audit.to_csv(ROOT / "results/sensor/channel_audit.csv", index=False)
pd.DataFrame(missingness).to_csv(ROOT / "results/missingness.csv", index=False)
pd.DataFrame(dates).to_csv(ROOT / "results/date_audit.csv", index=False)
alignment.to_csv(ROOT / "results/source_alignment.csv", index=False)
write_json(ROOT / "results/data_manifest.json", {
    "project": "Industrial-KG Fusion", "seed": config.SEED,
    "metropt3": {"source": config.METROPT_SOURCE, "download_url": config.METROPT_URL,
                 "rows": len(sensor), "sha256": sha256(metro_path), "license": "CC BY 4.0",
                 "median_interval_seconds": float(intervals.median())},
    "maintenance": {"source": config.MAINTENANCE_SOURCE, "rows": len(maintenance),
                    "sha256": sha256(maintenance_path), "snapshot": snapshot["maintenance_origin"]},
    "valid": valid, "invalid": invalid
})